# File Selection Analysis for File-Based Split
**Goal:** Determine which file is best for validation in file-based splitting strategy.

## Selection Criteria
1. **Band Coverage**: Has sufficient samples from all bands
2. **Distribution Match**: Similar distribution to overall training set
3. **Drift Diversity**: Contains varied drift scenarios
4. **Size**: Large enough for reliable validation metrics
5. **Difficulty**: Should be challenging but representative

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.band_mapping import load_band_mapping, classify_band, get_band_statistics
from utils.reproducibility_utils import seed_everything

SEED = 42
seed_everything(SEED)

TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
band_dict = load_band_mapping()

print(f"Train path: {TRAIN_PATH}")

## 1. Load and Analyze All Files

In [ ]:
# Load all files with detailed statistics
train_files = sorted(glob.glob(str(TRAIN_PATH / "train_part_*.csv")))
print(f"Found {len(train_files)} training files\n")

file_analysis = []

for f in train_files:
    df = pd.read_csv(f)
    filename = Path(f).name
    file_num = int(filename.split('_')[-1].replace('.csv', ''))
    
    # Band analysis
    bands = classify_band(df['sideSlip'].values, band_dict)
    
    # Feature statistics
    stats = {
        'file': filename,
        'file_num': file_num,
        'total_rows': len(df),
        # Band counts
        'many_count': (bands == 'many').sum(),
        'medium_count': (bands == 'medium').sum(),
        'few_count': (bands == 'few').sum(),
        # Band percentages
        'many_pct': 100 * (bands == 'many').sum() / len(bands),
        'medium_pct': 100 * (bands == 'medium').sum() / len(bands),
        'few_pct': 100 * (bands == 'few').sum() / len(bands),
        # Sideslip statistics
        'sideslip_mean': df['sideSlip'].mean(),
        'sideslip_std': df['sideSlip'].std(),
        'sideslip_abs_max': df['sideSlip'].abs().max(),
        'sideslip_abs_mean': df['sideSlip'].abs().mean(),
        # Drift intensity (proportion of high sideslip)
        'drift_intensity': (df['sideSlip'].abs() > 2.0).sum() / len(df) * 100,
        # Sensor diversity
        'yawrate_std': df['yawRate'].std(),
        'aycg_std': df['ayCG'].std(),
        'vxcg_mean': df['vxCG'].mean(),
    }
    file_analysis.append(stats)

file_df = pd.DataFrame(file_analysis)
print("File Analysis Summary:")
print(file_df[['file', 'total_rows', 'many_pct', 'medium_pct', 'few_pct', 'drift_intensity']].to_string(index=False))

## 2. Calculate Overall Distribution (Target)

In [ ]:
# Calculate overall distribution to match
all_data = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True)
overall_bands = classify_band(all_data['sideSlip'].values, band_dict)

overall_dist = {
    'many': 100 * (overall_bands == 'many').sum() / len(overall_bands),
    'medium': 100 * (overall_bands == 'medium').sum() / len(overall_bands),
    'few': 100 * (overall_bands == 'few').sum() / len(overall_bands)
}

print("Target Distribution (Overall):")
print(f"  Many:   {overall_dist['many']:.2f}%")
print(f"  Medium: {overall_dist['medium']:.2f}%")
print(f"  Few:    {overall_dist['few']:.2f}%")

## 3. Score Files by Selection Criteria

In [ ]:
# Calculate scores for each file
for idx, row in file_df.iterrows():
    # 1. Distribution match score (lower = better, closer to overall)
    dist_mismatch = (abs(row['many_pct'] - overall_dist['many']) + 
                     abs(row['medium_pct'] - overall_dist['medium']) + 
                     abs(row['few_pct'] - overall_dist['few']))
    file_df.loc[idx, 'dist_mismatch'] = dist_mismatch
    
    # 2. Few-shot coverage (we want some few-shot samples)
    few_coverage = row['few_count'] / max(file_df['few_count'].max(), 1) * 100
    file_df.loc[idx, 'few_coverage'] = few_coverage
    
    # 3. Size score (relative to largest)
    size_score = row['total_rows'] / file_df['total_rows'].max() * 100
    file_df.loc[idx, 'size_score'] = size_score
    
    # 4. Drift diversity (want varied drift scenarios)
    drift_diversity = row['sideslip_std'] / file_df['sideslip_std'].max() * 100
    file_df.loc[idx, 'drift_diversity'] = drift_diversity

# Calculate overall selection score (weighted combination)
file_df['selection_score'] = (
    100 - file_df['dist_mismatch'] * 2 +  # Penalize distribution mismatch
    file_df['few_coverage'] * 1.5 +        # Reward few-shot coverage
    file_df['size_score'] * 0.5 +           # Some weight to size
    file_df['drift_diversity'] * 1.0        # Reward drift diversity
)

# Rank files
file_df['rank'] = file_df['selection_score'].rank(ascending=False).astype(int)

In [ ]:
# Display ranking
ranking_cols = ['rank', 'file', 'total_rows', 'many_pct', 'medium_pct', 'few_pct', 
                'dist_mismatch', 'drift_intensity', 'selection_score']
ranked_df = file_df.sort_values('rank')[ranking_cols]
print("File Ranking for Validation Selection:")
print(ranked_df.to_string(index=False))

## 4. Visualize File Characteristics

In [ ]:
# Band distribution by file
fig = go.Figure()

for band, color in [('many_pct', 'green'), ('medium_pct', 'orange'), ('few_pct', 'red')]:
    fig.add_trace(go.Bar(
        x=file_df['file_num'],
        y=file_df[band],
        name=band.replace('_pct', ''),
        marker_color=color
    ))

# Add target distribution lines
fig.add_hline(y=overall_dist['many'], line_dash='dash', line_color='green', 
              annotation_text='Target Many')
fig.add_hline(y=overall_dist['medium'], line_dash='dash', line_color='orange')
fig.add_hline(y=overall_dist['few'], line_dash='dash', line_color='red')

fig.update_layout(
    title='Band Distribution by File (Dashed = Target)',
    xaxis_title='File Number',
    yaxis_title='Percentage',
    barmode='stack',
    height=500
)
fig.show()

In [ ]:
# Distribution mismatch by file
sorted_df = file_df.sort_values('dist_mismatch')

fig = go.Figure(go.Bar(
    x=sorted_df['file_num'],
    y=sorted_df['dist_mismatch'],
    marker_color=['green' if x < 10 else 'orange' if x < 20 else 'red' 
                  for x in sorted_df['dist_mismatch']],
    text=[f"{x:.1f}" for x in sorted_df['dist_mismatch']],
    textposition='outside'
))
fig.update_layout(
    title='Distribution Mismatch by File (Lower = Better Match)',
    xaxis_title='File Number',
    yaxis_title='Mismatch Score',
    height=400
)
fig.show()

## 5. Drift Intensity Analysis

In [ ]:
# Drift intensity per file
fig = go.Figure(go.Bar(
    x=file_df['file_num'],
    y=file_df['drift_intensity'],
    marker_color=px.colors.sequential.Reds,
    text=[f"{x:.1f}%" for x in file_df['drift_intensity']],
    textposition='outside'
))
fig.update_layout(
    title='Drift Intensity by File (% samples with |sideslip| > 2°)',
    xaxis_title='File Number',
    yaxis_title='Drift Intensity (%)',
    height=400
)
fig.show()

In [ ]:
# Files with highest drift content
high_drift = file_df.nlargest(5, 'drift_intensity')[['file', 'drift_intensity', 'few_count', 'sideslip_abs_max']]
print("Top 5 Files with Highest Drift Content:")
print(high_drift.to_string(index=False))

# Files with lowest drift (mostly normal driving)
low_drift = file_df.nsmallest(5, 'drift_intensity')[['file', 'drift_intensity', 'few_count', 'sideslip_abs_max']]
print("\nTop 5 Files with Lowest Drift Content:")
print(low_drift.to_string(index=False))

## 6. Sideslip Distribution per File

In [ ]:
# Sample files for detailed distribution view
sample_files = [1, 7, 10, 17]  # Select a few interesting files

fig = make_subplots(rows=len(sample_files), cols=1, 
                    subplot_titles=[f'File {n}' for n in sample_files])

for i, file_num in enumerate(sample_files, 1):
    filepath = TRAIN_PATH / f"train_part_{file_num:03d}.csv"
    if filepath.exists():
        df = pd.read_csv(filepath)
        fig.add_trace(
            go.Histogram(x=df['sideSlip'], nbinsx=100, name=f'File {file_num}', showlegend=False),
            row=i, col=1
        )

fig.update_layout(height=200*len(sample_files), title='Sideslip Distribution for Sample Files')
fig.show()

## 7. Feature Correlation Analysis per File

In [ ]:
# Check if files have different sensor characteristics
sensor_cols = ['yawRate', 'ayCG', 'handwheelAngle', 'vxCG', 'axCG']

feature_stats = []
for f in train_files:
    df = pd.read_csv(f)
    filename = Path(f).name
    file_num = int(filename.split('_')[-1].replace('.csv', ''))
    
    stats = {'file_num': file_num}
    for col in sensor_cols:
        stats[f'{col}_mean'] = df[col].mean()
        stats[f'{col}_std'] = df[col].std()
    feature_stats.append(stats)

feature_df = pd.DataFrame(feature_stats)

# Plot sensor means
fig = make_subplots(rows=2, cols=3, subplot_titles=sensor_cols + [''])
for i, col in enumerate(sensor_cols):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    fig.add_trace(
        go.Scatter(x=feature_df['file_num'], y=feature_df[f'{col}_mean'], 
                   mode='lines+markers', name=col, showlegend=False),
        row=row, col=col_idx
    )
    fig.update_yaxes(title_text=col, row=row, col=col_idx)

fig.update_layout(height=500, title='Sensor Means by File (Looking for Anomalies)')
fig.show()

## 8. Recommended Validation Files

In [ ]:
# Final recommendations
print("=" * 60)
print("VALIDATION FILE RECOMMENDATIONS")
print("=" * 60)

# Best overall match
best_match = file_df.loc[file_df['dist_mismatch'].idxmin()]
print(f"\n1. BEST DISTRIBUTION MATCH: File {int(best_match['file_num'])}")
print(f"   Distribution mismatch: {best_match['dist_mismatch']:.2f}")
print(f"   Few-shot count: {int(best_match['few_count'])}")

# Best for few-shot coverage
best_few = file_df.loc[file_df['few_count'].idxmax()]
print(f"\n2. BEST FEW-SHOT COVERAGE: File {int(best_few['file_num'])}")
print(f"   Few-shot count: {int(best_few['few_count'])}")
print(f"   Distribution mismatch: {best_few['dist_mismatch']:.2f}")

# Highest overall score
best_overall = file_df.loc[file_df['selection_score'].idxmax()]
print(f"\n3. BEST OVERALL SCORE: File {int(best_overall['file_num'])}")
print(f"   Selection score: {best_overall['selection_score']:.2f}")
print(f"   Distribution mismatch: {best_overall['dist_mismatch']:.2f}")
print(f"   Few-shot count: {int(best_overall['few_count'])}")

# Top 3 recommendations
top3 = file_df.nlargest(3, 'selection_score')[['file_num', 'total_rows', 'dist_mismatch', 'few_count', 'selection_score']]
print(f"\n4. TOP 3 RECOMMENDED FILES:")
print(top3.to_string(index=False))

In [ ]:
# Final visualization - radar chart comparing top candidates
top_files = file_df.nlargest(5, 'selection_score')

categories = ['Distribution\nMatch', 'Few-shot\nCoverage', 'Size', 'Drift\nDiversity']

fig = go.Figure()

for _, row in top_files.iterrows():
    values = [
        100 - row['dist_mismatch'],  # Invert so higher is better
        row['few_coverage'],
        row['size_score'],
        row['drift_diversity']
    ]
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories,
        fill='toself',
        name=f"File {int(row['file_num'])}"
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Top 5 Validation File Candidates (Radar Comparison)',
    height=500
)
fig.show()

## Summary

| Criterion | Best File | Value |
|-----------|-----------|-------|
| Distribution Match | See above | Lowest mismatch |
| Few-shot Coverage | See above | Most few-shot samples |
| Overall Score | See above | Weighted combination |

### Key Findings
1. Files vary significantly in drift content
2. Some files have almost no few-shot samples (poor for validation)
3. Distribution match is important for reliable validation metrics
4. Recommended to use file with balanced characteristics for stable training